# NB-03: 3D Rotary Position Embeddings (RoPE)

## Learning Objectives
- Understand why video transformers need 3D RoPE instead of 1D — separate frequency bands for temporal, height, and width axes preserve spatial-temporal structure
- Trace the head_dim split into three unequal bands: f_dim=44, h_dim=42, w_dim=42 for head_dim=128, and understand why the temporal axis absorbs the rounding remainder
- See how precompute_freqs_cis uses complex numbers (torch.polar) to represent rotation angles, and why float64 precision is required

## Prerequisites
- **Prior notebooks:** NB-01 (sinusoidal embedding concept), NB-02 (head_dim = dim // num_heads)
- **Assumed concepts:** Complex numbers as 2D rotations, RoPE in 1D (standard LLM positional encoding), torch.view_as_complex / torch.view_as_real

## Concept Map
- precompute_freqs_cis_3d → called once during WanModel.__init__ to precompute all position frequencies (NB-08)
- rope_apply → called inside SelfAttention.forward to rotate q and k before attention (NB-04)
- 3D frequency grid assembly → used in WanModel.forward to build per-token position encoding from (F, H, W) grid (NB-08)

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios (up to 6 levels up).
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
# Source: importlib.util.spec_from_file_location
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import symbols used in this notebook
from diffsynth.models.wan_video_dit import precompute_freqs_cis, precompute_freqs_cis_3d, rope_apply

import torch
from einops import rearrange

print("Setup complete.")

## 1. The 1D RoPE Base Function

The `precompute_freqs_cis` function creates a lookup table of complex rotation angles — one row per position, one complex number per frequency pair in `head_dim // 2`.

**How it works:**
1. Compute frequency scales: `freqs[k] = 1 / (theta^(2k/dim))` — a geometric sequence from 1/1 down to `1/theta`
2. Form the outer product: `freqs[position, k] = position * freqs[k]` — angle for position `p` at frequency `k`
3. Call `torch.polar(ones, freqs)` — converts each angle to a unit complex number `e^(i * angle)`

**Why float64?** `torch.polar` requires float64 (double precision) inputs. The magnitude argument must be `float64`, so the angles must be too. The returned tensor is `complex64` (single-precision complex), which is sufficient for the downstream multiply in `rope_apply`.

Source: `diffsynth/models/wan_video_dit.py`, line 83

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 83
dim_1d = 44                            # example dimension (will be f_dim in 3D split)
end = 1024                             # max positions in lookup table
freqs_1d = precompute_freqs_cis(dim_1d, end)  # [1024, dim_1d//2] complex

assert freqs_1d.shape == torch.Size([end, dim_1d // 2]), \
    f"Expected [{end}, {dim_1d//2}], got {freqs_1d.shape}"
print(f"1D freqs shape: {freqs_1d.shape}")
print(f"dtype: {freqs_1d.dtype}")
print(f"Each position has {dim_1d // 2} complex rotation angles")
print(f"First position (p=0) magnitudes: {freqs_1d[0].abs()[:3]} (should all be 1.0)")
print(f"Second position (p=1) real parts: {freqs_1d[1].real[:3]}")

## 2. The Three-Axis Band Split

Video tokens have three positional axes: **temporal** (frame index F), **height** (spatial row H), and **width** (spatial column W). `precompute_freqs_cis_3d` creates three independent 1D frequency tables — one per axis — by splitting `head_dim` into three bands.

**The split is NOT equal thirds** (Pitfall 2 from RESEARCH.md). Integer floor division creates an asymmetry:
- `h_dim = head_dim // 3` — floor division (e.g., 128 // 3 = 42)
- `w_dim = head_dim // 3` — same as h_dim
- `f_dim = head_dim - 2 * (head_dim // 3)` — temporal absorbs the rounding remainder

For `head_dim = 128`: `h_dim = w_dim = 42`, `f_dim = 128 - 84 = 44`. The temporal axis gets 2 extra dimensions, not because it is more important but because integer arithmetic requires the remainder to go somewhere.

Source: `diffsynth/models/wan_video_dit.py`, line 75

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 75
head_dim = 128                         # dim=1536, num_heads=12 => 1536//12 = 128

# The three-axis split (NOT equal thirds — Pitfall 2):
f_dim = head_dim - 2 * (head_dim // 3)  # 128 - 2*42 = 44 (temporal absorbs rounding)
h_dim = head_dim // 3                   # 42
w_dim = head_dim // 3                   # 42
assert f_dim + h_dim + w_dim == head_dim, \
    f"{f_dim}+{h_dim}+{w_dim} != {head_dim}"
print(f"head_dim = {head_dim}")
print(f"f_dim (temporal) = {f_dim}  <- absorbs rounding remainder")
print(f"h_dim (height)   = {h_dim}")
print(f"w_dim (width)    = {w_dim}")
print(f"Sum: {f_dim} + {h_dim} + {w_dim} = {f_dim + h_dim + w_dim}")
print()
print(f"Complex pairs per axis:")
print(f"  f_dim//2 = {f_dim//2}  (temporal: 22 complex rotation angles per position)")
print(f"  h_dim//2 = {h_dim//2}  (height:   21 complex rotation angles per position)")
print(f"  w_dim//2 = {w_dim//2}  (width:    21 complex rotation angles per position)")
print(f"  total    = {f_dim//2 + h_dim//2 + w_dim//2}  (= head_dim//2 = {head_dim//2})")

## 3. Calling precompute_freqs_cis_3d

With the band arithmetic understood, we can call `precompute_freqs_cis_3d(head_dim)` to get three separate 1D frequency tables — one for each spatial-temporal axis. Each table has `end=1024` rows, covering up to 1024 positions per axis.

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 75
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)

# Each axis gets a separate 1D frequency table:
# shape = [end, axis_dim//2] where axis_dim//2 = number of complex values
assert f_freqs.shape == torch.Size([1024, f_dim // 2]), \
    f"Expected [1024, {f_dim//2}], got {f_freqs.shape}"
assert h_freqs.shape == torch.Size([1024, h_dim // 2]), \
    f"Expected [1024, {h_dim//2}], got {h_freqs.shape}"
assert w_freqs.shape == torch.Size([1024, w_dim // 2]), \
    f"Expected [1024, {w_dim//2}], got {w_freqs.shape}"
print(f"f_freqs: {f_freqs.shape}  (temporal: {f_dim//2} complex per position)")
print(f"h_freqs: {h_freqs.shape}  (height:   {h_dim//2} complex per position)")
print(f"w_freqs: {w_freqs.shape}  (width:    {w_dim//2} complex per position)")
print(f"dtype: {f_freqs.dtype}")

## 4. 3D Frequency Grid Assembly

To apply RoPE to video tokens, we need to build a **per-token** position encoding that reflects each token's position in the 3D grid `(F, H, W)`. The three separate 1D tables are assembled into a single tensor by broadcasting each axis table across the other two and concatenating along the frequency dimension.

The result has shape `[seq_len, 1, head_dim//2]` where `seq_len = F * H * W`. The `1` in the middle broadcasts over `num_heads` (all heads share the same positional encoding).

Source: `diffsynth/models/wan_video_dit.py`, line 381

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 381-385 (WanModel.forward)
F, H, W = 5, 4, 4                      # 5 frames, 4x4 spatial grid

freqs = torch.cat([
    f_freqs[:F].view(F, 1, 1, -1).expand(F, H, W, -1),  # [F,H,W,f_dim//2] temporal
    h_freqs[:H].view(1, H, 1, -1).expand(F, H, W, -1),  # [F,H,W,h_dim//2] height
    w_freqs[:W].view(1, 1, W, -1).expand(F, H, W, -1),  # [F,H,W,w_dim//2] width
], dim=-1).reshape(F * H * W, 1, -1)   # [seq_len, 1, head_dim//2]

seq_len = F * H * W                    # 80
assert freqs.shape == torch.Size([seq_len, 1, head_dim // 2]), \
    f"Expected [{seq_len}, 1, {head_dim//2}], got {freqs.shape}"
print(f"Assembled freqs: {freqs.shape}")
print(f"  seq_len = F*H*W = {F}*{H}*{W} = {seq_len}")
print(f"  head_dim//2 = {head_dim//2} = f_dim//2 + h_dim//2 + w_dim//2 = {f_dim//2}+{h_dim//2}+{w_dim//2}")
print(f"  dtype: {freqs.dtype}")

## 5. rope_apply — Dtype Handling and Float64

The `rope_apply` function rotates each query/key vector by the precomputed angles. The key implementation detail is **dtype management**:

1. Input `x` arrives as `float32` (or `bfloat16` during training)
2. `x.to(torch.float64)` upcasts before `view_as_complex` — complex multiplication requires matching dtypes between the float64-precision complex values from `torch.polar` and the input
3. The result is cast back to the original dtype with `.to(x.dtype)` at the end

The `view_as_complex` call interprets pairs of consecutive floats `(a, b)` as the complex number `a + bi`. This requires the last dimension to be even-sized — which is guaranteed because each axis band has an even `dim//2` frequency count.

**Shape requirement:** `rope_apply` expects `freqs` of shape `[S, 1, head_dim//2]` — the full concatenated frequency grid, not a single-axis band.

Source: `diffsynth/models/wan_video_dit.py`, line 92

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 92
B, S, dim, num_heads = 1, 20, 1536, 12
x = torch.randn(B, S, dim)             # [B, S, dim] — float32 input

# Build full 3D assembled freqs for S=20 tokens (e.g., F=5, H=2, W=2)
# freqs must have shape [S, 1, head_dim//2] — the full concatenated representation
F_s, H_s, W_s = 5, 2, 2               # 5*2*2 = 20 = S
freqs_rope = torch.cat([
    f_freqs[:F_s].view(F_s, 1, 1, -1).expand(F_s, H_s, W_s, -1),  # [F,H,W,f_dim//2]
    h_freqs[:H_s].view(1, H_s, 1, -1).expand(F_s, H_s, W_s, -1),  # [F,H,W,h_dim//2]
    w_freqs[:W_s].view(1, 1, W_s, -1).expand(F_s, H_s, W_s, -1),  # [F,H,W,w_dim//2]
], dim=-1).reshape(S, 1, -1)           # [S, 1, head_dim//2]

assert freqs_rope.shape == torch.Size([S, 1, head_dim // 2]), \
    f"Expected [{S}, 1, {head_dim//2}], got {freqs_rope.shape}"
print(f"Input x dtype:  {x.dtype}")   # float32
out = rope_apply(x, freqs_rope, num_heads)   # [B, S, dim]
print(f"Output dtype: {out.dtype}")    # float32 (restored)
assert out.shape == x.shape, \
    f"Expected {x.shape}, got {out.shape}"
assert out.dtype == x.dtype, \
    f"Expected {x.dtype}, got {out.dtype}"
print(f"rope_apply: {x.shape} -> {out.shape}")
print(f"dtype preserved: {x.dtype} -> {out.dtype}")
print(f"Note: internally upcasts to float64 for complex multiply, then casts back")

## Summary

### Key Takeaways
- **3D RoPE splits head_dim into three unequal bands**: f_dim=44 (temporal), h_dim=42 (height), w_dim=42 (width) for head_dim=128. The temporal axis absorbs the floor-division remainder — this is arithmetic necessity, not a design priority.
- **precompute_freqs_cis returns complex tensors**: `torch.polar` converts `(magnitude=1, angle)` pairs into unit complex numbers on the unit circle. Each position maps to `dim//2` rotation angles. The function uses float64 internally because `torch.polar` requires it.
- **Grid assembly broadcasts three 1D tables into one 3D grid**: Each axis table is view-expanded to `[F, H, W, axis_dim//2]` and the three are concatenated to form `[seq_len, 1, head_dim//2]`.
- **rope_apply preserves input dtype**: It upcasts to float64 for the complex multiplication, then casts the result back to the original dtype (float32 or bfloat16). Freqs must be the full `[S, 1, head_dim//2]` concatenated grid.

### Source References
| Symbol | Location |
|--------|----------|
| precompute_freqs_cis_3d | `diffsynth/models/wan_video_dit.py`, line 75 |
| precompute_freqs_cis | `diffsynth/models/wan_video_dit.py`, line 83 |
| rope_apply | `diffsynth/models/wan_video_dit.py`, line 92 |
| 3D grid assembly | `diffsynth/models/wan_video_dit.py`, line 381 |

## Exercises

### Exercise 1 — Different head_dim
Change `head_dim` from 128 to 64 in Cell 6. Verify that `f_dim = 64 - 2*(64//3) = 22`, `h_dim = w_dim = 21`. Does the assertion `f_dim + h_dim + w_dim == head_dim` still pass? Confirm the `precompute_freqs_cis_3d(64)` output shapes match the new band sizes.

### Exercise 2 — Change theta
Change `theta` from 10000.0 to 1000.0 by calling `precompute_freqs_cis(dim_1d, end, theta=1000.0)` in Cell 4. Compare `freqs_1d[100].real[:3]` with the original theta=10000 values. What happens to the frequency range when theta decreases? (Hint: lower theta means faster oscillation — positions go through more rotations over a given range.)

### Exercise 3 — Remove float64 upcast
In the `rope_apply` source (line 92), the code does `.to(torch.float64)` before `view_as_complex`. In a test cell, try: `torch.view_as_complex(torch.randn(2, 2))`. What error do you get? Why does PyTorch require float64 (not float32) for `view_as_complex` when used with `torch.polar` output?